In [1]:
%%bash
cat > matrixAdd.cu << 'EOF'
#include <stdio.h>
#include <cuda_runtime.h>
#include <math.h>
#include <time.h>

#define MATRIX_SIZE 2048
#define BLOCK_SIZE 16

__global__ void matrixAddKernel(int *d_A, int *d_B, int *d_C, int rows, int cols) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < rows && col < cols) {
        int idx = row * cols + col;
        d_C[idx] = d_A[idx] + d_B[idx];
    }
}

void matrixAddCPU(int *h_A, int *h_B, int *h_C, int rows, int cols) {
    for (int i = 0; i < rows; i++) {
        for (int j = 0; j < cols; j++) {
            int idx = i * cols + j;
            h_C[idx] = h_A[idx] + h_B[idx];
        }
    }
}

bool verifyResults(int *gpu_result, int *cpu_result, int size) {
    for (int i = 0; i < size; i++) {
        if (abs(gpu_result[i] - cpu_result[i]) > 0) {
            return false;
        }
    }
    return true;
}

int main() {
    printf("========== CUDA MATRIX ADDITION PROGRAM ==========\n\n");

    int rows = MATRIX_SIZE;
    int cols = MATRIX_SIZE;
    size_t matrix_bytes = rows * cols * sizeof(int);

    printf("PROBLEM SETUP:\n");
    printf("==============\n");
    printf("Matrix dimensions: %d x %d\n", rows, cols);
    printf("Matrix size: %.2f MB per matrix\n", matrix_bytes / (1024.0f * 1024.0f));
    printf("Total GPU memory: %.2f MB\n\n", (3 * matrix_bytes) / (1024.0f * 1024.0f));

    printf("STEP 1: ALLOCATING HOST MEMORY\n");
    printf("===============================\n");
    int *h_A = (int *)malloc(matrix_bytes);
    int *h_B = (int *)malloc(matrix_bytes);
    int *h_C_GPU = (int *)malloc(matrix_bytes);
    int *h_C_CPU = (int *)malloc(matrix_bytes);

    if (h_A == NULL || h_B == NULL || h_C_GPU == NULL || h_C_CPU == NULL) {
        printf("Error: Memory allocation failed!\n");
        return 1;
    }
    printf("Host memory allocated successfully\n\n");

    printf("STEP 2: INITIALIZING INPUT MATRICES\n");
    printf("====================================\n");
    srand(time(NULL));

    for (int i = 0; i < rows * cols; i++) {
        h_A[i] = rand() % 100;
        h_B[i] = rand() % 100;
    }

    printf("Matrices initialized with random values (0-99)\n\n");

    printf("STEP 3: ALLOCATING DEVICE MEMORY\n");
    printf("=================================\n");
    int *d_A = NULL;
    int *d_B = NULL;
    int *d_C = NULL;

    cudaMalloc((void **)&d_A, matrix_bytes);
    cudaMalloc((void **)&d_B, matrix_bytes);
    cudaMalloc((void **)&d_C, matrix_bytes);

    printf("Device memory allocated successfully\n\n");

    printf("STEP 4: COPYING MATRICES HOST TO DEVICE\n");
    printf("========================================\n");
    cudaMemcpy(d_A, h_A, matrix_bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, matrix_bytes, cudaMemcpyHostToDevice);
    printf("Matrices copied to device\n\n");

    printf("STEP 5: SETTING UP GRID AND BLOCK DIMENSIONS\n");
    printf("=============================================\n");

    dim3 block_dim(BLOCK_SIZE, BLOCK_SIZE, 1);
    dim3 grid_dim((cols + BLOCK_SIZE - 1) / BLOCK_SIZE,
                  (rows + BLOCK_SIZE - 1) / BLOCK_SIZE, 1);

    printf("Block dimensions: (%d, %d, %d)\n", block_dim.x, block_dim.y, block_dim.z);
    printf("Grid dimensions: (%d, %d, %d)\n", grid_dim.x, grid_dim.y, grid_dim.z);
    printf("Total threads: %d\n\n", (grid_dim.x * grid_dim.y) * (block_dim.x * block_dim.y));

    printf("STEP 6: LAUNCHING CUDA KERNEL\n");
    printf("==============================\n");

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start, 0);
    matrixAddKernel<<<grid_dim, block_dim>>>(d_A, d_B, d_C, rows, cols);
    cudaEventRecord(stop, 0);
    cudaEventSynchronize(stop);

    float elapsed_time_ms = 0.0f;
    cudaEventElapsedTime(&elapsed_time_ms, start, stop);

    printf("Kernel executed successfully\n");
    printf("Execution time: %.3f ms\n\n", elapsed_time_ms);

    printf("STEP 7: COPYING RESULTS DEVICE TO HOST\n");
    printf("========================================\n");
    cudaMemcpy(h_C_GPU, d_C, matrix_bytes, cudaMemcpyDeviceToHost);
    printf("Results copied to host\n\n");

    printf("STEP 8: VERIFYING RESULTS\n");
    printf("==========================\n");
    printf("Computing CPU reference...\n");
    matrixAddCPU(h_A, h_B, h_C_CPU, rows, cols);

    printf("Comparing GPU and CPU results...\n");
    if (verifyResults(h_C_GPU, h_C_CPU, rows * cols)) {
        printf(">>> RESULTS VERIFIED: CORRECT! <<<\n\n");
    } else {
        printf(">>> RESULTS VERIFIED: INCORRECT! <<<\n\n");
    }

    printf("STEP 9: FREEING DEVICE MEMORY\n");
    printf("==============================\n");
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    printf("Device memory freed\n\n");

    printf("========== PERFORMANCE ANALYSIS ==========\n\n");

    long long total_operations = (long long)rows * cols;

    printf("1. FLOATING POINT OPERATIONS (FLOPs):\n");
    printf("   Total matrix elements: %d x %d = %lld\n", rows, cols, total_operations);
    printf("   Operations per element: 1\n");
    printf("   >>> TOTAL FLOPs: %lld (%.2e) <<<\n\n", total_operations, (double)total_operations);

    printf("2. GLOBAL MEMORY READS:\n");
    printf("   Data per thread: 2 elements (A[i][j], B[i][j])\n");
    printf("   Total threads: %lld\n", total_operations);
    printf("   >>> TOTAL READS: %lld x 2 = %lld (%.2e) <<<\n", total_operations, 2 * total_operations, (double)(2 * total_operations));
    printf("   Total bytes read: %.2f MB\n\n", (2.0 * total_operations * 4) / (1024.0 * 1024.0));

    printf("3. GLOBAL MEMORY WRITES:\n");
    printf("   Data per thread: 1 element (C[i][j])\n");
    printf("   Total threads: %lld\n", total_operations);
    printf("   >>> TOTAL WRITES: %lld x 1 = %lld (%.2e) <<<\n", total_operations, total_operations, (double)total_operations);
    printf("   Total bytes written: %.2f MB\n\n", (1.0 * total_operations * 4) / (1024.0 * 1024.0));

    printf("4. MEMORY TRAFFIC SUMMARY:\n");
    long long total_bytes = (2 * total_operations + total_operations) * 4;
    printf("   Total bytes transferred: %.2f MB\n", total_bytes / (1024.0 * 1024.0));
    printf("   Bandwidth: %.2f GB/s\n\n", (total_bytes / (1024.0 * 1024.0 * 1024.0)) / (elapsed_time_ms / 1000.0));

    printf("5. SAMPLE RESULTS (5x5):\n");
    for (int i = 0; i < 5; i++) {
        for (int j = 0; j < 5; j++) {
            int idx = i * cols + j;
            printf("[%d,%d]: %d + %d = %d  ", i, j, h_A[idx], h_B[idx], h_C_GPU[idx]);
        }
        printf("\n");
    }
    printf("\n==========================================\n");

    free(h_A);
    free(h_B);
    free(h_C_GPU);
    free(h_C_CPU);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}
EOF

In [2]:
%%bash
nvcc -Wno-deprecated-gpu-targets matrixAdd.cu -o matrixAdd
./matrixAdd

========== CUDA MATRIX ADDITION PROGRAM ==========

PROBLEM SETUP:
Matrix dimensions: 2048 x 2048
Matrix size: 16.00 MB per matrix
Total GPU memory: 48.00 MB

STEP 1: ALLOCATING HOST MEMORY
Host memory allocated successfully

STEP 2: INITIALIZING INPUT MATRICES
Matrices initialized with random values (0-99)

STEP 3: ALLOCATING DEVICE MEMORY
Device memory allocated successfully

STEP 4: COPYING MATRICES HOST TO DEVICE
Matrices copied to device

STEP 5: SETTING UP GRID AND BLOCK DIMENSIONS
Block dimensions: (16, 16, 1)
Grid dimensions: (128, 128, 1)
Total threads: 4194304

STEP 6: LAUNCHING CUDA KERNEL
Kernel executed successfully
Execution time: 89.240 ms

STEP 7: COPYING RESULTS DEVICE TO HOST
Results copied to host

STEP 8: VERIFYING RESULTS
Computing CPU reference...
Comparing GPU and CPU results...
>>> RESULTS VERIFIED: CORRECT! <<<

STEP 9: FREEING DEVICE MEMORY
Device memory freed

========== PERFORMANCE ANALYSIS ==========

1. FLOATING POINT OPERATIONS (FLOPs):
   Total matrix el